# Computer Lab: Dispersion and Chromaticity — local Python version

This notebook replaces the original Sirepo/Elegant workflow with a fully local first-order matrix model. The companion file `dc_local_helpers.py` contains the transfer matrices, matching routines, plotting code, and widget wrappers. The notebook cells expose only the physics knobs students should inspect or modify.

The model is intentionally pedagogical: drifts, thick quadrupoles, sector bends, linear dispersion, finite-difference chromaticity, and tune-footprint plots. It is not a production lattice model.

<style>
.answer {
    color: #b00020;
    border-left: 4px solid #b00020;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.answer table, .answer th, .answer td {
    color: #b00020;
}
.instructor-note {
    color: #7a3b00;
    border-left: 4px solid #7a3b00;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.note {
    background-color: #f6f8fa;
    padding: 0.75em;
    border-left: 4px solid #999;
}
</style>


## Learning goals

By the end of this lab you should be able to:

1. Explain how a bend creates horizontal dispersion.
2. Use \(\sigma_x^2=\epsilon_x\beta_x+\eta_x^2\sigma_\delta^2\) to estimate beam-size growth from momentum spread.
3. Tune a two-bend section to make an achromat.
4. Distinguish local zero dispersion from globally small dispersion.
5. Estimate momentum acceptance from both aperture and chromatic resonance crossing.
6. Interpret a tune footprint on a low-order resonance diagram.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

import dc_local_helpers as dch
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

pd.set_option("display.precision", 6)
display(dch.check_environment())


## Model conventions

All distances are in meters. Quadrupole strength \(k_1\) is in \(\mathrm{m}^{-2}\). Positive \(k_1\) focuses horizontally and defocuses vertically. The beam has geometric emittance

\[
\epsilon_x=\epsilon_y=6\ \mathrm{mm\,mrad}=6\times 10^{-6}\ \mathrm{m\,rad}.
\]

The horizontal vector used internally is \([x,x',\delta]\), where \(\delta=\Delta p/p_0\). Bends include first-order dispersion source terms. The aperture exercise uses an rms-envelope proxy, so the aperture condition is explicitly

\[
n_\sigma\sigma_x = R.
\]

The default is \(n_\sigma=1\) to match the simplified style of the original lab. You can change it later.


In [ ]:
emit = dch.GEOMETRIC_EMITTANCE
sigma_delta = 1e-3       # 0.1% momentum spread
pipe_radius_m = 0.025    # 2.5 cm
n_sigma_aperture = 1.0   # use 1 for the original simplified rms criterion


# A. Dispersion in a FODO lattice

We start from a 5 m FODO cell. The reference case has no bend. The modified case replaces the 0.5 m bend slot with a 20-degree sector bend. This is the local-code replacement for editing the FODO lattice in Sirepo.


## A0. Reference matched FODO cell with no dipole

Run the baseline matched cell. Record the beam sizes at QF and QD. These are your comparison values before adding a dipole.


In [ ]:
fodo_reference = dch.make_fodo_cell(kq=0.6, with_bend=False)
optics_fodo_reference = dch.compute_periodic_optics(fodo_reference)

display(dch.element_layout(fodo_reference))
display(dch.stability_report(fodo_reference))
display(dch.optics_summary(optics_fodo_reference, label="reference FODO"))
display(dch.beam_size_comparison_at_elements(optics_fodo_reference, ["QF", "QD"], sigma_delta=0.0))


In [ ]:
dch.plot_optics(optics_fodo_reference, title="Reference matched FODO cell: no bend")
dch.plot_beam_size(optics_fodo_reference, sigma_delta=0.0, title="Reference FODO beam size, σδ = 0")


## A1. Replace the bend slot with a dipole

Now insert a 20-degree bend. The default model below uses a sector bend without edge focusing, so the main new effect is horizontal dispersion. The next short comparison lets you optionally turn edge focusing on and separate it from the dispersive size term.


In [ ]:
fodo_with_bend = dch.make_fodo_cell(kq=0.6, with_bend=True, bend_angle_deg=20.0, edge_focusing=False)
optics_fodo_bend = dch.compute_periodic_optics(fodo_with_bend)

display(dch.element_layout(fodo_with_bend))
display(dch.stability_report(fodo_with_bend))
display(dch.optics_summary(optics_fodo_bend, label="FODO with bend"))

dch.plot_optics(optics_fodo_bend, title="Matched FODO cell with a 20-degree bend")


In [ ]:
# Optional comparison: turn on a simple thin-edge focusing model.
fodo_with_edges = dch.make_fodo_cell(kq=0.6, with_bend=True, bend_angle_deg=20.0, edge_focusing=True)
optics_fodo_edges = dch.compute_periodic_optics(fodo_with_edges)

comparison = pd.concat(
    [
        dch.optics_summary(optics_fodo_reference, "no bend"),
        dch.optics_summary(optics_fodo_bend, "bend, no edge focusing").drop(columns="quantity"),
        dch.optics_summary(optics_fodo_edges, "bend + edge focusing").drop(columns="quantity"),
    ],
    axis=1,
)
display(comparison)


## Q1. Minimum dispersion and beam-size comparison

Find the minimum horizontal dispersion in the FODO cell with the bend. At what element or element region does it occur: focusing quadrupole, defocusing quadrupole, drift, or bend?

Then compare the no-bend and bend cases. Which beam-size differences come from dispersion, and which differences can occur even at \(\sigma_\delta=0\) because the bend changed the optics?


In [ ]:
display(dch.dispersion_extrema(optics_fodo_bend))

display(dch.beam_size_comparison_at_elements(optics_fodo_reference, ["QF", "QD"], sigma_delta=0.0))
display(dch.beam_size_comparison_at_elements(optics_fodo_bend, ["QF", "QD"], sigma_delta=0.0))


**Your Q1 answer**

Minimum \(\eta_x\) =  
Location =  
Beam-size comparison with the no-bend FODO =  
Which differences are dispersive? Which are optical/edge-focusing differences?


## Q2. Beam size with 0.1% momentum spread

Assume

\[
\sigma_\delta=\frac{\Delta p}{p_0}=0.001.
\]

At QF, compute the expected horizontal beam size from

\[
\sigma_x = \sqrt{\epsilon_x\beta_x + (\eta_x\sigma_\delta)^2}.
\]

Compare with the \(\sigma_\delta=0\) result. Then do the same comparison for \(\sigma_y\).


In [ ]:
sigma_delta = 1e-3

display(dch.table_at_element_centers(optics_fodo_bend, ["QF"], sigma_delta=sigma_delta))
display(dch.beam_size_comparison_at_elements(optics_fodo_bend, ["QF"], sigma_delta=sigma_delta))

dch.plot_beam_size(optics_fodo_bend, sigma_delta=sigma_delta, title="FODO with bend: effect of 0.1% momentum spread")


**Your Q2 answer**

| quantity at QF | \(\sigma_\delta=0\) | \(\sigma_\delta=0.001\) |
|---|---:|---:|
| \(\sigma_x\) | | |
| \(\sigma_y\) | | |

Explain why the horizontal and vertical sizes behave differently.


## Try it: bend angle and momentum spread

Move the bend-angle and \(\sigma_\delta\) sliders. Watch how \(\eta_x\), \(\sigma_x\), and \(\sigma_y\) respond. This replaces the old Sirepo workflow of editing the lattice and rerunning with a new `Sigma DP` value.


In [ ]:
dch.interactive_fodo_dispersion()


# B. Designing a zero-dispersion insert

The next model is a compact two-bend achromat-like section with two 18-degree bends. With all quadrupoles off, a beam that starts with zero dispersion does not end with zero dispersion. We will use quadrupoles to make \(\eta_x\) and \(\eta_x'\) return to zero.


## B0. Two-bend cell with all quadrupoles off

Use transport optics, not periodic optics, for this first part. The initial conditions are

\[
\beta_x=\beta_y=10\ \mathrm{m},\quad \alpha_x=\alpha_y=0,\quad \eta_x=\eta_x'=0.
\]


In [ ]:
two_bend_off = dch.make_dba_cell(q1=0.0, q2=0.0, q3=0.0)
optics_two_bend_off = dch.compute_transport_optics(two_bend_off)

display(dch.element_layout(two_bend_off))
display(dch.dba_endpoint_table(q1=0.0, q2=0.0, q3=0.0))
dch.plot_optics(optics_two_bend_off, title="Two-bend section with all quadrupoles off")


## Q3. End-of-cell dispersion before correction

Record \(\eta_x\) and \(\eta_x'\) at the end of the uncorrected cell. Does this cell qualify as achromatic at its exit?


In [ ]:
eta_end, etap_end = dch.endpoint_dispersion(two_bend_off)
pd.DataFrame({"quantity": ["end eta_x [m]", "end eta_x'"], "value": [eta_end, etap_end]})


**Your Q3 answer**

\(\eta_x(s_\mathrm{end})\) =  
\(\eta_x'(s_\mathrm{end})\) =  
Achromatic at the end? Why or why not?


## B1. Scan Q1 to cancel outgoing dispersion

The central quadrupole Q1 changes the dispersion trajectory between the two bends. First inspect the scan. Then use the optimizer result as a check.


In [ ]:
q1_scan = dch.scan_q1_for_achromat(qmin=0.0, qmax=6.0, n=301)
display(q1_scan.sort_values("penalty").head(10))
dch.plot_q1_scan(q1_scan, title="Manual scan: endpoint dispersion versus Q1")


In [ ]:
q1_achromat = dch.best_q1_for_achromat(qmin=0.0, qmax=6.0)

display(dch.dba_endpoint_table(q1=q1_achromat, q2=0.0, q3=0.0))
print(f"Best Q1 from local scan/optimizer: {q1_achromat:.6f} 1/m^2")


## Q4. Q1 strength for approximately zero outgoing dispersion

Report the Q1 strength you found, with at least two decimal places.

**Your Q4 answer**

\(k_{1,\mathrm{Q1}} =\)


## Try it: tune the achromat quadrupole by hand

Move Q1 and watch the final values of \(\eta_x\) and \(\eta_x'\). The best setting is where both endpoint values are near zero.


In [ ]:
dch.interactive_achromat_q1()


## B2. Add focusing so the cell can be repeated

Canceling outgoing dispersion in an open beamline does not automatically make a stable periodic cell. The default focused DBA-like cell keeps the same Q1 achromat condition and adds two flanking quadrupoles Q2 and Q3. Their placement is outside the zero-dispersion insert, so they can change periodic focusing while preserving the local achromat condition.


In [ ]:
central_q1_cell = dch.make_dba_cell(q1=q1_achromat, q2=0.0, q3=0.0)
focused_dba_cell = dch.make_dba_cell(q1=q1_achromat, q2=dch.DBA_Q2_DEFAULT, q3=dch.DBA_Q3_DEFAULT)

print("Central-Q1-only cell stability:")
display(dch.stability_report(central_q1_cell))

print("Focused DBA-like cell stability:")
display(dch.stability_report(focused_dba_cell))

display(dch.dba_endpoint_table(q1=q1_achromat, q2=dch.DBA_Q2_DEFAULT, q3=dch.DBA_Q3_DEFAULT))

optics_focused_dba = dch.compute_periodic_optics(focused_dba_cell)
display(dch.optics_summary(optics_focused_dba, label="focused DBA cell"))
dch.plot_optics(optics_focused_dba, title="Stable focused DBA-like periodic cell")


## Q5. Maximum dispersion in the focused DBA-like cell

A dispersion-free insert can still have a dispersion bump elsewhere. Report the maximum \(\eta_x\) in the stable focused cell and its location.


In [ ]:
display(dch.dispersion_extrema(optics_focused_dba))
display(dch.table_at_element_centers(optics_focused_dba, ["Q1", "B1", "B2", "Q2", "Q3"], sigma_delta=0.0))


**Your Q5 answer**

Maximum \(\eta_x\) =  
Location =  
Why this does not contradict “zero-dispersion insert”:


## Momentum acceptance limited by dispersion

With a finite pipe radius, momentum spread can make the horizontal envelope too large. The simplified criterion used here is

\[
n_\sigma\sqrt{\epsilon_x\beta_x+\eta_x^2\sigma_\delta^2}=R.
\]

The original lab effectively used the \(n_\sigma=1\) rms proxy. You can later change `n_sigma_aperture` to test a stricter engineering convention.


## Q6. Momentum-spread limit from a 2.5 cm pipe radius

Using \(R=2.5\ \mathrm{cm}\) and \(n_\sigma=1\), find the largest \(\sigma_\delta\) that can be transported before the horizontal rms envelope reaches the pipe radius.


In [ ]:
aperture_limits = dch.aperture_limit_table(
    optics_focused_dba,
    pipe_radius_m=pipe_radius_m,
    n_sigma=n_sigma_aperture,
)
display(aperture_limits.head(10))

dispersion_limit = float(aperture_limits.iloc[0]["max_sigma_delta"])
dispersion_limit_location = str(aperture_limits.iloc[0]["element"])
print(f"Dispersion/aperture limit: sigma_delta = {dispersion_limit:.6f} ({100*dispersion_limit:.3f}%)")
print(f"Limiting element: {dispersion_limit_location}")


**Your Q6 answer**

Largest tolerable \(\sigma_\delta\) =  
Equivalent percent momentum spread =  
Limiting element/location =


## Q7. Where does beam loss occur first?

If the momentum spread exceeds your Q6 value, identify where the horizontal envelope first reaches or exceeds the aperture. Is it simply the place with the largest \(\eta_x\), or does \(\beta_x\) also matter?


In [ ]:
sigma_delta_test = 1.10 * dispersion_limit

dch.plot_beam_size_with_aperture(
    optics_focused_dba,
    sigma_delta=sigma_delta_test,
    pipe_radius_m=pipe_radius_m,
    n_sigma=n_sigma_aperture,
)

display(dch.aperture_summary(optics_focused_dba, pipe_radius_m=pipe_radius_m, n_sigma=n_sigma_aperture))


**Your Q7 answer**

Loss/worst-envelope location =  
Reason =


## Try it: aperture and momentum spread

Use the sliders to compare different momentum spreads and different aperture criteria. In particular, compare \(n_\sigma=1\), 3, and 5.


In [ ]:
dch.interactive_aperture(optics_focused_dba)


# C. Chromaticity in a ring

Now repeat the stable DBA-like cell 10 times to form a model ring. A ring has betatron tunes \(Q_x\) and \(Q_y\). Off-momentum particles see different effective focusing, so their tunes shift with \(\delta\). This derivative is chromaticity.


In [ ]:
n_ring_cells = 10
ring_cell = focused_dba_cell
ring = dch.repeat_cell(ring_cell, n_cells=n_ring_cells)

ring_info = dch.ring_summary(ring_cell, n_cells=n_ring_cells)
display(ring_info)

dch.plot_optics(optics_focused_dba, title="One cell of the repeated DBA-like ring")


## Q8. Record the x and y tunes

Record \(Q_x\) and \(Q_y\) to three significant figures.


In [ ]:
Qx, Qy = dch.ring_tunes(ring_cell, n_cells=n_ring_cells)
print(f"Qx = {Qx:.6f}")
print(f"Qy = {Qy:.6f}")


**Your Q8 answer**

\(Q_x=\)  
\(Q_y=\)


## Q9. Why are the tunes not necessarily equal?

In the earlier no-bend FODO reference case, the horizontal and vertical phase advances were equal because the lattice was symmetric between the two planes. Inspect the beta functions in the DBA-like cell and explain why equal tunes should not be expected here.


In [ ]:
dch.plot_optics(optics_focused_dba, title="Beta functions and dispersion in one DBA-like cell")
display(dch.optics_summary(optics_focused_dba, label="one DBA-like cell"))


**Your Q9 answer**

Reason the horizontal and vertical tunes need not be equal:


## Q10. Tune spread for 0.1% momentum spread

Use

\[
\Delta Q_x=C_x\sigma_\delta,\qquad \Delta Q_y=C_y\sigma_\delta.
\]

For this question use \(\sigma_\delta=0.001\).


In [ ]:
Cx, Cy = dch.chromaticity_finite_difference(ring_cell, n_cells=n_ring_cells)
chromatic_table = dch.chromatic_spread_table(Qx, Qy, Cx, Cy, sigma_delta=1e-3)
display(chromatic_table)


**Your Q10 answer**

\(C_x=\)  
\(C_y=\)  
\(\Delta Q_x\) for \(\sigma_\delta=0.001\) =  
\(\Delta Q_y\) for \(\sigma_\delta=0.001\) =


## Resonance diagram and chromatic tune footprint

A resonance line satisfies

\[
mQ_x+nQ_y=p,
\]

where \(m\), \(n\), and \(p\) are integers. The resonance order is \(|m|+|n|\). In this exercise, assume resonances of order 3 and below are forbidden, while order 4 and above are tolerable.


In [ ]:
# Start with the 0.1% momentum spread used in Q10.
dch.plot_tune_footprint(Qx, Qy, Cx, Cy, sigma_delta=1e-3, resonance_order=3)


## Try it: move the chromatic tune footprint

Increase \(\sigma_\delta\) until the tune-footprint line first crosses a resonance of order 3 or lower. Compare your visual estimate with the table in Q11.


In [ ]:
dch.interactive_tune_footprint(Qx, Qy, Cx, Cy)


## Q11. Momentum acceptance from chromatic resonance crossing

Using resonances through order 3, estimate the largest \(\sigma_\delta\) that can be tolerated before the chromatic tune footprint hits a forbidden resonance. Report the value to the nearest 0.1% if using the graph, and compare to the algorithmic value in the table.


In [ ]:
resonance_limits = dch.first_resonance_crossing(Qx, Qy, Cx, Cy, max_order=3)
display(resonance_limits.head(10))

chromaticity_limit = float(resonance_limits.iloc[0]["abs_delta_at_crossing"])
nearest_resonance = str(resonance_limits.iloc[0]["resonance"])
print(f"Chromaticity/resonance limit: sigma_delta = {chromaticity_limit:.6f} ({100*chromaticity_limit:.3f}%)")
print(f"Nearest forbidden resonance: {nearest_resonance}")


**Your Q11 answer**

Largest tolerable \(\sigma_\delta\) from the resonance diagram =  
Nearest forbidden resonance =


## Q12. Compare dispersion and chromaticity limits

Compare the aperture/dispersion limit from Q6 with the chromaticity/resonance limit from Q11. The smaller value sets the momentum acceptance in this simplified model.


In [ ]:
comparison = dch.acceptance_comparison(
    dispersion_limit_delta=dispersion_limit,
    chromatic_limit_delta=chromaticity_limit,
)
display(comparison)


**Your Q12 answer**

The stricter momentum-acceptance limit is:  
Reason:


# Optional extension exercises

1. Repeat Q6 with `n_sigma_aperture = 3`. Does the aperture limit become more or less restrictive than the chromaticity limit?
2. Change the FODO bend angle in Section A. How does \(\eta_x\) scale with bend angle for small angles?
3. Change Q2 and Q3 in the DBA cell using `dch.interactive_dba_stability()`. Find a stable cell with smaller maximum \(\beta_y\).
4. Change the number of ring cells. What happens to total tune and chromaticity?
5. Add edge focusing in the FODO or DBA builders. Which conclusions change, and which remain the same?
